Goal: Use a ridge and lasso model to predict the ClosePrice of real estate. 
Ridge and Lasso utilizes regularization, which is adding a penalty to large coefficients so the model cannot overfit. Ridge is ideal when features are highly correlated and when less variance, more generalizaiton is needed. Lasso regression penalizes coefficients based on their absolute magnitutde. Lasso is ideal when only a few predictors matter. Ridge shrinks weights, Lasso shrinks and removes useless features. 

In [2]:
#final_traning.csv: training set (cleaned + processed)
#oct_testing.csv: holdout/final test set
#steps: load two datasets, separate x & y, scale features using training data, 
#fit ridge & lasso, evaluate on both train + test

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import GridSearchCV

In [3]:
train = pd.read_csv("final_training.csv")
test = pd.read_csv("oct_testing.csv")

In [4]:
target = "ClosePrice"
numeric_cols = ['Latitude', 'Longitude', 'LivingArea', 'ParkingTotal', 'YearBuilt', 'BathroomsTotalInteger',
                'BedroomsTotal', 'Stories', 'PropertyAgeAtClose', 'GarageSpaces']

In [5]:
X_train = train[numeric_cols]
y_train = train[target]

X_test = test[numeric_cols]
y_test = test[target]

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
#Ridge hyperparameter tuning
# 4. Ridge hyperparameter tuning
ridge_params = {"alpha": [0.1, 1, 5, 10, 20, 50, 100]}
ridge = GridSearchCV(Ridge(), ridge_params, cv=5)
ridge.fit(X_train_scaled, y_train)

# 5. Lasso hyperparameter tuning
lasso_params = {"alpha": [0.001, 0.01, 0.1, 1, 5, 10]}
lasso = GridSearchCV(Lasso(max_iter=5000), lasso_params, cv=5)
lasso.fit(X_train_scaled, y_train)

# 6. Predictions
ridge_pred = ridge.predict(X_test_scaled)
lasso_pred = lasso.predict(X_test_scaled)

/Users/zoeyip/Desktop/idx-exchange/venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.897e+14, tolerance: 6.494e+12
  model = cd_fast.enet_coordinate_descent(
/Users/zoeyip/Desktop/idx-exchange/venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.789e+16, tolerance: 6.673e+12
  model = cd_fast.enet_coordinate_descent(
/Users/zoeyip/Desktop/idx-exchange/venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the fe

In [8]:
def evaluate_model(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    print(f"\n{name} Results")
    print("R²:", r2)
    print("MAPE:", mape)
    print("MdAPE:", mdape)

# 8. Evaluate both models
evaluate_model("Ridge (best alpha = {})".format(ridge.best_params_['alpha']),
               y_test, ridge_pred)

evaluate_model("Lasso (best alpha = {})".format(lasso.best_params_['alpha']),
               y_test, lasso_pred)


Ridge (best alpha = 20) Results
R²: 0.48968706628268477
MAPE: 40.83532372424208
MdAPE: 28.17595509651275

Lasso (best alpha = 0.001) Results
R²: 0.4897304381614064
MAPE: 40.837838644243206
MdAPE: 28.215185857148985


My linear regression model: 
R²: 0.087
MAPE: 54.62
MdAPE: 36.15

Ridge & Lasso Model: R² = 0.49
My model explains about half of the variation in closing prices. Adding regularizatino helped reduce overiftting from unscaled features. This model performs better within the limited feature space. 

MAPE = 40%
On average, my model is off by 40% of the true price. 
Good relative to original approach (54%), but bad in an absolute sense. Need to compare with other models, as this is not really an acceptable statistic for a real estate pricing model. 

MdAPE = 28%
Half of predictions are within 28% of actual price. More trustworthy than MAPE. 

Overall, dataset may not include enough information to hit strong accuracy. However, the ridge & lasso model has significant improvements compared to the original baseline linear regression model. 